In [1]:
# %pip install --quiet --upgrade pip
# %pip install numpy --quiet
# %pip install Pandas --quiet
# %pip install sklearn --quiet
# %pip install ipywidgets --quiet

# Horse Racing Stats Builder

Using the data derived from the features extracted by the [Feature Analysis](https://github.com/LeeSanderson/RacingData/blob/main/Data/FeatureAnalysis.ipynb) notebook, create a database (CSV file) of Horse stats.


In [ ]:
import os
import numpy as np
import pandas as pd
import math
from abc import ABC, abstractmethod
from datetime import datetime, date

from race_analytics.data_path import get_data_path

_DATA_PATH = get_data_path()

In [3]:
races = pd.read_csv(os.path.join(_DATA_PATH, "Race_Features.csv"))
races["Off"] = pd.to_datetime(races["Off"], format="%Y-%m-%d %H:%M:%S")

In [4]:
races = races.sort_values(["HorseId", "Off"], ascending=[True, False])
horse_races = races.groupby("HorseId").first().reset_index()

In [5]:
horse_races["NumberOfPriorRaces"] = horse_races["NumberOfPriorRaces"].fillna(0)
horse_races["AvgRelFinishingPosition"] = (
    (horse_races["LastRaceAvgRelFinishingPosition"] * horse_races["NumberOfPriorRaces"])
    + (horse_races["FinishingPosition"] / horse_races["HorseCount"])
) / (horse_races["NumberOfPriorRaces"] + 1)

horse_races["AvgRelFinishingPosition"] = horse_races["AvgRelFinishingPosition"].fillna(
    (horse_races["FinishingPosition"] / horse_races["HorseCount"])
)
horse_races["NumberOfPriorRaces"] = horse_races["NumberOfPriorRaces"] + 1

In [6]:
# horse_races[["HorseId", "AvgRelFinishingPosition", "LastRaceAvgRelFinishingPosition", "FinishingPosition", "HorseCount", "NumberOfPriorRaces"]]

In [7]:
horse_stats = horse_races[
    [
        "HorseId",
        "Off",
        "DistanceInMeters",
        "WeightInPounds",
        "AvgRelFinishingPosition",
        "Surface_AllWeather",
        "Surface_Dirt",
        "Surface_Turf",
        "Going_Firm",
        "Going_Good",
        "Going_Good_To_Firm",
        "Going_Good_To_Soft",
        "Going_Heavy",
        "Going_Soft",
        "RaceType_Flat",
        "RaceType_Hurdle",
        "RaceType_Other",
        "RaceType_SteepleChase",
        "Speed",
    ]
].rename(
    columns={
        "Off": "LastOff",
        "DistanceInMeters": "LastRaceDistanceInMeters",
        "WeightInPounds": "LastRaceWeightInPounds",
        "AvgRelFinishingPosition": "LastRaceAvgRelFinishingPosition",
        "Surface_AllWeather": "LastRaceSurface_AllWeather",
        "Surface_Dirt": "LastRaceSurface_Dirt",
        "Surface_Turf": "LastRaceSurface_Turf",
        "Going_Firm": "LastRaceGoing_Firm",
        "Going_Good": "LastRaceGoing_Good",
        "Going_Good_To_Firm": "LastRaceGoing_Good_To_Firm",
        "Going_Good_To_Soft": "LastRaceGoing_Good_To_Soft",
        "Going_Heavy": "LastRaceGoing_Heavy",
        "Going_Soft": "LastRaceGoing_Soft",
        "RaceType_Flat": "LastRaceRaceType_Flat",
        "RaceType_Hurdle": "LastRaceRaceType_Hurdle",
        "RaceType_Other": "LastRaceRaceType_Other",
        "RaceType_SteepleChase": "LastRaceRaceType_SteepleChase",
        "Speed": "LastRaceSpeed",
    }
)

In [8]:
horse_stats.to_csv(os.path.join(_DATA_PATH, "Horse_Stats.csv"), index=False)